In [1]:
import torch
from torch import autograd

In [ ]:
class FlashAttentionFunction(autograd.Function):
    @staticmethod
    def forward(ctx, 
                q: torch.Tensor, 
                k: torch.Tensor, 
                v: torch.Tensor, 
                is_causal: bool =False) -> torch.Tensor:
        batch_size = q.shape[0]
        q_seq_len = q.shape[1]
        n_dim = q.shape[2]
        kv_seq_len = k.shape[1]

        q_tile_size = 16
        q_rows = q_seq_len // q_tile_size 
        kv_tile_size = 16
        kv_rows = kv_seq_len // kv_tile_size
        o = torch.zeros_like(q) 
        l = torch.zeros([batch_size, q_seq_len])

        for batch_idx in range(batch_size):
            for i in range(q_rows):
                q_i = q[batch_idx, i * q_tile_size : (i + 1) * q_tile_size] # [q_tile_size, n_dim]
                o_i = torch.zeros_like(q_i)   # [q_tile_size, n_dim]
                l_i = torch.zeros(q_tile_size)    # [q_tile_size]
                m_i_last = torch.tensor([-float('inf')] * q_tile_size)    # [q_tile_size]
                for j in range(kv_rows):
                    k_j = k[batch_idx, j * kv_tile_size : (j + 1) * kv_tile_size]  # [kv_tile_size, n_dim]
                    v_j = v[batch_idx, j * kv_tile_size : (j + 1) * kv_tile_size]  # [kv_tile_size, n_dim]
                    q_scale = n_dim ** 0.5 
                    s_ij = q_i @ k_j.T / q_scale  # [q_tile_size, kv_tile_size]
                    m_ij = torch.max(s_ij, dim = -1, keepdim=False)[0] # [q_tile_size]
                    m_i = torch.max(torch.stack([m_i_last, m_ij], dim = -1), dim=-1, keepdim=False)[0] # [q_tile_size]
                    p_ij = torch.exp(s_ij - m_i[:, None]) #  [q_tile_size, kv_tile_size] softmax的分子
                    l_i = l_i * torch.exp(m_i_last - m_i) + torch.sum(p_ij, dim=-1, keepdim=False) # [q_tile_size] softmax分母的部分和
                    o_i = torch.diag(torch.exp(m_i_last - m_i)) @ o_i + p_ij @ v_j # [q_tile_size, n_dim]
                    m_i_last = m_i  
                o[batch_idx, i * q_tile_size : (i + 1) * q_tile_size] = torch.diag(1.0 / l_i) @ o_i # [q_tile_size, n_dim]
                l[batch_idx, i * q_tile_size : (i + 1) * q_tile_size] = m_i_last + torch.log(l_i) # [q_tile_size]

        ctx.save_for_backward(q, k, v, o, l)
        ctx.is_causal = is_causal
        
        return o                







In [94]:
impl = FlashAttentionFunction.apply 
batch_size = 1
n_queries = 3
n_keys = 5
D = 10
q = torch.randn(batch_size, n_queries, D,  requires_grad=True)
k = torch.randn(batch_size, n_keys, D, requires_grad=True)
v = torch.randn(batch_size, n_keys, D, requires_grad=True)

In [95]:
a = impl(q, k, v)
a

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]],
       grad_fn=<FlashAttentionFunctionBackward>)

In [77]:
# o_i = torch.zeros_like(q_i)   # [q_tile_size, n_dim]
# l_i = torch.zeros(q_tile_size)    # [q_tile_size]
# m_i_last = torch.tensor([-1e9] * q_tile_size)    # [q_tile_size]
#q_i = q[batch_idx, i * q_tile_size : (i + 1) * q_tile_size] # [q_tile_size, n_dim]
# k_j = k[batch_idx, j * kv_tile_size : (j + 1) * kv_tile_size]  # [kv_tile_size, n_dim]
# v_j = v[batch_idx, j * kv_tile_size : (j + 1) * kv_tile_size]  # [kv_tile_size, n_dim]
# q_scale = n_dim ** 0.5 
# s_ij = q_i @ k_j.T / q_scale  # [q_tile_size, kv_tile_size]
# m_ij = torch.max(s_ij, dim = -1, keepdim=False)[0] # [q_tile_size]
# m_i = torch.max(torch.stack([m_i_last, m_ij], dim = -1), dim=-1, keepdim=False)[0] # [q_tile_size]
# p_ij = torch.exp(s_ij - m_i[:, None]) #  [q_tile_size, kv_tile_size] softmax的分子
# l_i = l_i * torch.exp(m_i_last - m_i) + torch.sum(p_ij, dim=-1, keepdim=False) # [q_tile_size] softmax分母的部分和
# o_i = torch.diag(torch.exp(m_i_last - m_i)) @ o_i + p_ij @ v_j # [q_tile_size, n_dim]
# m_i_last = m_i
o_i = torch.zeros([3, 10])
l_i = torch.zeros(3)
q_i = torch.randn([3, 10])
k_j = torch.randn([5, 10]) 
q_scale = 10 ** 0.5 
s_ij = q_i @ k_j.T / q_scale
s_ij
m_i_last = torch.tensor([-float('inf')] * 3) 
m_i_last
m_ij = torch.max(s_ij, dim = -1, keepdim=False)[0] # [q_tile_size]
m_ij
m_i = torch.max(torch.stack([m_i_last, m_ij], dim = -1), dim=-1, keepdim=False)[0] # [q_tile_size]
m_i
p_ij = torch.exp(s_ij - m_i[:, None]) #  [q_tile_size, kv_tile_size] softmax的分子 
p_ij 
l_i = l_i * torch.exp(m_i_last - m_i) + torch.sum(p_ij, dim=-1, keepdim=False) 
l_i 
v_j = torch.randn(5, 10)
o_i = torch.diag(torch.exp(m_i_last - m_i)) @ o_i + p_ij @ v_j # [q_tile_size, n_dim] 
o_i
# s_ij.shape 
# m_i.shape


tensor([[ 0.5671,  0.2034, -0.1701, -0.0711,  2.0376, -0.2202, -0.8474, -0.2518,
          1.0328,  0.9426],
        [ 0.9365, -0.7615, -0.8411,  0.6061,  2.4730,  0.5520, -0.8834, -0.2790,
          2.3334,  1.5473],
        [ 2.9267, -0.5231, -0.2068,  1.2620,  1.1984,  2.0689, -1.0016, -0.6152,
          2.7863, -0.1329]])

In [65]:
s_ij.shape 

torch.Size([3, 5])

In [66]:
l_i = torch.zeros(3) 
p_ij = torch.exp(s_ij - m_i[:, None])
l_i = l_i * torch.exp(m_i_last - m_i) + torch.sum(p_ij, dim=-1, keepdim=False) 
l_i

tensor([1.9677, 2.5505, 3.4980])